# 01.8 — Feature Selection: rdmoldes 50-descriptor set

**Goal**: select a global (endpoint-agnostic) subset of the paper's 50 rdmoldes descriptors
(316 expanded features).

**Scope**: selection runs on the `rdkit` featureset only (rdmoldes, 316 features) —
FCFP4/ECFP4 fingerprints are a separate featureset and are not touched by this pipeline.

**Granularity**: selection decisions are made per *descriptor* (one of the 50 named
descriptors — 44 single-feature scalars + 6 multi-feature vector descriptors like
`AUTOCORR2D` -> 192 features), never per individual expanded feature. See
`src/feature_selection/`.

**Pipeline**:
1. Drop degenerate (near-constant) descriptors
2. Reduce each surviving descriptor to its top-k principal components (needed to compare
   multi-feature descriptors without collapsing them to one coincidental axis)
3. Mutual information per descriptor, per endpoint -> max across endpoints ("keep if useful for any endpoint")
4. Correlation prune (drop near-duplicate descriptors via canonical correlation, MI tiebreak)
5. VIF prune (drop multicollinear descriptors, MI tiebreak)
6. LightGBM recursive descriptor elimination -> inspect the CV-score trace, pick a cutoff by eye

Selected once globally (all 4 modelling endpoints); revisit per-endpoint only if one endpoint
performs significantly worse after this.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import mutual_info_regression  # used by section 1b's raw-feature diagnostic

from src.feature_selection import (
    rdmoldes_descriptor_map,
    drop_constant_descriptors,
    descriptor_component_matrix,
    descriptor_pc1_matrix,
    mutual_info_per_descriptor,
    correlation_prune,
    vif_prune,
    vif_mi_table,
    evaluate_descriptor_set,
    run_descriptor_rfe,
)

RANDOM_STATE = 42
DATA_PROC = Path('../data/processed')
FIGURES = Path('../figures')

ENDPOINTS_MODEL = ['HLM', 'MDR1', 'SOL', 'RLM']  # PPB_H/PPB_R excluded from modelling (too sparse)
N_COMPONENTS = 5  # top-k PCs per descriptor, for MI + correlation-pruning (not VIF, which stays PC1-only)

feat = joblib.load(DATA_PROC / 'section3_feat.pkl')
for ep in ENDPOINTS_MODEL:
    print(f"{ep:6s}  N={feat[ep]['rdkit'].shape[0]:5d}  rdkit_features={feat[ep]['rdkit'].shape[1]}")

# X_by_endpoint/y_by_endpoint_arr: each endpoint's own real rows (no dedup, no pooling) --
# used everywhere a real model actually gets fit (R^2 audits below, RFE in section 5).
X_by_endpoint = {ep: feat[ep]['rdkit'] for ep in ENDPOINTS_MODEL}
y_by_endpoint_arr = {ep: feat[ep]['y'] for ep in ENDPOINTS_MODEL}

## 1 — Descriptor map + constant-descriptor filter

Steps 1-4 (variance, MI, correlation, VIF) only ever look at the descriptor features
themselves (`X`), not any endpoint's target — descriptor redundancy is a property of the
molecules' structures, not of a specific endpoint. Many compounds are tested across more
than one of the 4 endpoints, so naively row-stacking all 4 endpoints' matrices would count
those molecules 2-4x when estimating variance/correlation/VIF, giving them outsized
influence on what "typical" redundancy looks like. Instead we pool the **unique molecules**
only (deduplicated by canonical SMILES) for steps 1-4. Step 6 (RFE) and the MI step still
use each endpoint's own full (non-deduplicated) rows against its own target — that's
real data, not an estimation-bias concern.

A descriptor is "**constant**" if every one of its features has ~zero variance across all
molecules — i.e. it returns essentially the same value for every compound, so it can never
help distinguish one molecule from another.

In [ ]:
# Deduplicate by canonical SMILES across the 4 endpoints -- each unique molecule
# contributes exactly one row to X_all, regardless of how many endpoints tested it.
smiles_to_gid = {}
X_unique_rows = []
endpoint_gids = {}  # ep -> global ids for that endpoint's own rows, in that endpoint's own order
for ep in ENDPOINTS_MODEL:
    gids = np.empty(len(feat[ep]['smiles']), dtype=int)
    for i, smi in enumerate(feat[ep]['smiles']):
        if smi not in smiles_to_gid:
            smiles_to_gid[smi] = len(X_unique_rows)
            X_unique_rows.append(feat[ep]['rdkit'][i])
        gids[i] = smiles_to_gid[smi]
    endpoint_gids[ep] = gids
X_all = np.array(X_unique_rows)

n_endpoint_rows = sum(feat[ep]['rdkit'].shape[0] for ep in ENDPOINTS_MODEL)
print(f'{len(smiles_to_gid)} unique molecules from {n_endpoint_rows} endpoint-rows '
      f'({n_endpoint_rows - len(smiles_to_gid)} cross-endpoint duplicates removed)')
print(f'X_all shape: {X_all.shape}')

descriptor_map = rdmoldes_descriptor_map()
print(f'{len(descriptor_map)} descriptors, {sum(len(f) for f in descriptor_map.values())} total features')

kept_map, dropped_constant = drop_constant_descriptors(X_all, descriptor_map)
print(f'Dropped {len(dropped_constant)} constant descriptor(s): {dropped_constant}')
print(f'{len(kept_map)} descriptors remain')

# CalcPBF and CalcSpherocityIndex are exactly 0.0 for every molecule -- and this IS a direct,
# deterministic consequence of the flat z=0 conformers (ADR-011), not an unrelated RDKit quirk.
# Both formulas trivially evaluate to 0 when every atom has z=0: PBF is the mean deviation of
# atoms from their own best-fit plane (already exactly the xy-plane when z=0), and RDKit's
# Spherocity Index is 3*lambda_min/(lambda1+lambda2+lambda3) over the eigenvalues of the atomic
# coordinate covariance matrix -- lambda_min is exactly 0 when the z-column has zero variance.
# CalcAsphericity/CalcEccentricity stay non-constant because they use the mass-weighted
# inertia tensor instead, whose two in-plane principal moments are still generically distinct
# for a non-linear planar molecule (verified: rdkit.Chem.rdMolDescriptors on AddHs('CCO') gives
# PBF=0.0, Spherocity=0.0, but Eccentricity=0.993, Asphericity=0.695, PMI1=10.6 (not 0)).
# Correctly dropped either way; noted here in case either descriptor is ever relied on
# elsewhere in the project.
for name in dropped_constant:
    print(f'  {name}: unique value = {np.unique(X_all[:, descriptor_map[name]])}')

baseline_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()), random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- post constant-filter ({len(kept_map)} descriptors): {baseline_r2}')

## 1b — Diagnostic: would raw-feature-level selection be more interpretable?

Steps 2-5 below make every keep/drop decision at the **descriptor** level -- a 192-feature
block like `AUTOCORR2D` is kept or dropped whole, never split across its own columns. The
alternative would be selecting directly among the 316 raw rdmoldes feature columns instead
of the 50 named descriptors. That would remove the descriptor-level machinery entirely (no
PCA reduction, no CCA -- with `k=1` on both sides, canonical correlation between two raw
scalar columns is just `|Pearson r|`) and would let VIF operate on real, uncompressed
columns instead of a single PC1 axis.

But it trades that off against interpretability: does raw-feature-level selection actually
surface named, chemist-readable descriptors, or would it just grab scattered individual
bins out of the least interpretable blocks? Checked directly below with per-column MI (no
PCA, no descriptor grouping) on all 316 raw features.

In [ ]:
mi_per_col = {}
for ep in ENDPOINTS_MODEL:
    mi_per_col[ep] = mutual_info_regression(X_by_endpoint[ep], y_by_endpoint_arr[ep], random_state=RANDOM_STATE)

col_to_desc = {col: name for name, cols in descriptor_map.items() for col in cols}

mi_col_df = pd.DataFrame(mi_per_col)
mi_col_df['max_mi'] = mi_col_df.max(axis=1)
mi_col_df['descriptor'] = [col_to_desc[i] for i in range(X_all.shape[1])]

top30 = mi_col_df.sort_values('max_mi', ascending=False).head(30)
print('Top 30 raw features by per-column MI (max across the 4 endpoints, no PCA/grouping):')
print(top30[['descriptor', 'max_mi']].to_string())
print('\nDescriptor family counts in the top 30:')
print(top30['descriptor'].value_counts())

autocorr_hits = top30.index[top30['descriptor'] == 'AUTOCORR2D']
autocorr_positions = sorted(descriptor_map['AUTOCORR2D'].index(i) for i in autocorr_hits)
print(f"\n{len(autocorr_hits)}/30 top raw features come from AUTOCORR2D alone -- individually"
      f" unnamed, positions {autocorr_positions[0]}-{autocorr_positions[-1]} of 192, not"
      f" clustered in a small interpretable sub-range.")
print("Conclusion: raw-feature-level selection would pull in a large fraction of anonymous"
      " AUTOCORR2D bins alongside named scalars. Descriptor-level selection (used for the"
      " rest of this notebook) trades away some potential dimensionality precision for"
      " guaranteed chemist-interpretable output -- kept as the primary methodology on that"
      " basis.")

### 1b visualization — top raw features by MI (N=30 / 15 / 5)

The printed diagnostic above showed AUTOCORR2D supplying 17/30 of the top raw columns by
per-column MI, scattered (not clustered) across its 192 bins. As a chart: three panels,
ranking all 316 raw rdmoldes feature columns by max-MI (max across the 4 endpoints, no
PCA/grouping — same `mi_col_df` computed above), showing the top 30, top 15, and top 5.
Bars are colored by parent descriptor and labeled with the descriptor name (plus the
column's position within its own block, for multi-feature descriptors like AUTOCORR2D or
PEOE_VSA, so an anonymous vector bin is still traceable back to its source).

Narrowing N=30 -> 15 -> 5 is a direct visual of the interpretability argument above: as
the list shortens, it converges on a small set of *named, whole* descriptors (`CalcChi0n`,
`CalcNumLipinskiHBA`, `CalcLabuteASA`, ...) rather than staying dominated by individual
vector-descriptor bins — no single AUTOCORR2D/MQNs/VSA *column* is anywhere near
competitive with the strongest scalars on its own. This is also why the descriptor-level
pipeline's final answer (§9: `PEOE_VSA`+`SlogP_VSA`) looks like it's in tension with this
panel — it isn't: `PEOE_VSA`/`SlogP_VSA`'s predictive power is a *joint*, block-level
property of their 14/12 columns acting together (§7b/§9), invisible to any single column's
MI score. Individual vector columns aren't very useful; whole vector *descriptors* can
still be the most useful thing in the entire pool.

In [ ]:
import matplotlib.cm as cm

def _feature_label(descriptor, col_idx):
    positions = descriptor_map[descriptor]
    if len(positions) == 1:
        return descriptor
    return f'{descriptor}[{positions.index(col_idx)}]'

mi_sorted = mi_col_df.sort_values('max_mi', ascending=False)
top30 = mi_sorted.head(30)
families = sorted(top30['descriptor'].unique())  # only families that actually appear in the top 30
palette = {name: cm.tab20(i % 20) for i, name in enumerate(families)}

fig, axes = plt.subplots(1, 3, figsize=(15, 7))
for ax, n in zip(axes, [30, 15, 5]):
    top_n = mi_sorted.head(n).iloc[::-1]  # reverse so highest MI plots at the top
    labels = [_feature_label(row.descriptor, idx) for idx, row in top_n.iterrows()]
    colors = [palette[d] for d in top_n['descriptor']]
    ax.barh(labels, top_n['max_mi'], color=colors, edgecolor='white')
    ax.set_title(f'Top {n} raw features by MI')
    ax.set_xlabel('max MI (nats), across 4 endpoints')
    ax.tick_params(axis='y', labelsize=8)

# No shared legend -- each bar's y-axis label already names its descriptor (plus its
# position within the block, for multi-feature descriptors); color only reinforces which
# bars share a parent descriptor across the three panels, it doesn't carry new information.
fig.suptitle('Raw-feature-level MI: individual vector-descriptor columns are not, on their own, very informative')
fig.tight_layout()
fig.savefig(FIGURES / 'section_fs_raw_mi_top_features.png', dpi=150, bbox_inches='tight')
plt.show()


## 2 — Descriptor components + mutual information

Each descriptor reduces to its top `N_COMPONENTS` principal components (fewer if the
descriptor has fewer features — the 44 scalar descriptors always reduce to 1).
Correlation-pruning and MI use this full component set rather than a single PC1, so a
large descriptor like `AUTOCORR2D` (192 features) isn't judged by one
coincidentally-informative-or-not axis. MI is computed per endpoint (each endpoint has its
own non-NaN row subset), taking the max over (component, endpoint) per descriptor — a
descriptor is kept if it's useful for *any* endpoint.

In [ ]:
component_map = descriptor_component_matrix(X_all, kept_map, n_components=N_COMPONENTS, random_state=RANDOM_STATE)
print(f'{len(component_map)} descriptors; components per descriptor (min/max): '
      f'{min(c.shape[1] for c in component_map.values())}/{max(c.shape[1] for c in component_map.values())}')

# y indexed by each endpoint's own global molecule ids (not a contiguous range) --
# mutual_info_per_descriptor positionally indexes component_map by these, so a molecule
# shared across endpoints correctly reuses the same descriptor row for each endpoint's own MI calc.
y_by_endpoint = {}
for ep in ENDPOINTS_MODEL:
    y_by_endpoint[ep] = pd.Series(feat[ep]['y'], index=endpoint_gids[ep])

mi_max, mi_table = mutual_info_per_descriptor(component_map, y_by_endpoint, random_state=RANDOM_STATE)
mi_table['max'] = mi_max
mi_table['mi_rank'] = mi_max.rank(ascending=False)

print('\nMutual information (nats) per descriptor, per endpoint -- HLM/MDR1/SOL/RLM columns are '
      "MI(descriptor's best component, that endpoint's target); 'max' is the max across the 4 "
      "endpoints (the score used everywhere downstream); 'mi_rank' ranks descriptors by that max, "
      "most useful first.")
mi_table.sort_values('max', ascending=False)

## 3 — Correlation prune

Drop one descriptor from each near-duplicate pair (canonical correlation between their
top-component sets > 0.9), keeping whichever has the higher max-MI.

**Why CCA here and not just PCA again**: PCA (step 2) answers a *within*-descriptor
question — "what are this descriptor's own dominant axes of variation." Canonical
Correlation Analysis (CCA) answers a *between*-descriptor question — "how correlated are
these two descriptors with each other," using all of each one's top components jointly
rather than only comparing their first components to each other. It's the standard
generalisation of Pearson correlation from two single variables to two multi-dimensional
variable sets — exactly what's needed to judge whether two descriptors are redundant
without arbitrarily privileging their dominant axis.

**Known limitation**: CCA only finds the strongest *linear* combination correlation
between two descriptors' components — genuinely nonlinear (but real) dependence between a
pair could be missed, letting a redundant pair survive as "not correlated." A
distance-correlation-style measure (zero if and only if truly independent, linear or not)
would close this gap, at the cost of a new dependency and O(n²) compute over ~3500
molecules. Not adopted here: VIF and the RFE/R² audit downstream both re-check whatever
this step decides against real model performance, so a wrong call here isn't silently
trusted — see `_canonical_correlation`'s docstring in `src/feature_selection/`.

In [ ]:
CORR_THRESHOLD = 0.9

kept_after_corr, corr_dropped = correlation_prune(component_map, mi_max, threshold=CORR_THRESHOLD)
print(f'Dropped {len(corr_dropped)} descriptor(s) on correlation:')
for rec in corr_dropped:
    print(f"  {rec['dropped']:28s} (kept {rec['kept_instead']:28s} instead, canonical_corr={rec['canonical_corr']:.3f})")
print(f'{len(kept_after_corr)} descriptors remain')

corr_stage_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, kept_after_corr, random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- post correlation-prune ({len(kept_after_corr)} descriptors): {corr_stage_r2}')
print(f'  -> mean R^2: {corr_stage_r2["mean"]:.4f}  (was {baseline_r2["mean"]:.4f} before any pruning)')

## 4 — VIF prune

First, a combined VIF+MI table for every block still standing — so a high VIF and a high
MI can be viewed side by side before any automated elimination happens (this is what
`vif_prune` below does automatically, one block at a time, but seeing the whole table at
once makes it easy to sanity-check that automation isn't cutting something important).

Then: iteratively drop the highest-VIF block (multicollinearity against all other
surviving blocks, not just a pairwise check) until every remaining block is at or below
the threshold. Near-tied VIFs are broken by MI.

In [ ]:
pc1_df = descriptor_pc1_matrix(component_map)  # VIF is inherently single-variable-vs-the-rest; stays PC1-only
vif_mi_table(pc1_df[kept_after_corr], mi_max)

In [ ]:
VIF_THRESHOLD = 5.0

# Static mi_rank among the candidate pool VIF starts from (kept_after_corr) -- printed
# alongside each drop below. vif_elim_rank isn't included in that per-drop print: VIF is
# recomputed every iteration as descriptors leave, so "rank" only has a fixed meaning at
# the very start (see the vif_mi_table above for that snapshot) -- past the first
# iteration it would need re-deriving from vif_prune's internals, not worth the complexity.
mi_rank_at_vif_start = mi_max[kept_after_corr].rank(ascending=False)

kept_after_vif, vif_trace = vif_prune(pc1_df[kept_after_corr], mi_max, threshold=VIF_THRESHOLD)
print(f'Dropped {len(vif_trace)} descriptor(s) on VIF:')
for rec in vif_trace:
    print(f"  {rec['dropped']:28s} VIF={rec['vif']:8.2f}  MI={rec['mi']:.4f}  (mi_rank={mi_rank_at_vif_start[rec['dropped']]:.0f}/{len(kept_after_corr)})  ({rec['remaining']} left)")
print(f'{len(kept_after_vif)} descriptors remain: {kept_after_vif}')

vif_stage_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, kept_after_vif, random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- before VIF (={len(kept_after_corr)}-descriptor post-correlation stage): {corr_stage_r2}')
print(f'Stage R^2 audit -- after VIF  ({len(kept_after_vif)} descriptors):                         {vif_stage_r2}')
print(f'  -> mean R^2: {vif_stage_r2["mean"]:.4f}  (was {corr_stage_r2["mean"]:.4f} after correlation-prune, {baseline_r2["mean"]:.4f} before any pruning)')

## 5 — LightGBM recursive descriptor elimination

Operates on each endpoint's own full features (not PC1) for the surviving descriptors. At
each step, fits LightGBM per endpoint, scores each descriptor by its **summed** feature
gain-importance across its own columns (max across endpoints — same "useful for any
endpoint" rule), and eliminates the lowest-scoring descriptor. The CV score is logged at
every step so a bad elimination shows up as a visible drop in the trace rather than being
silently accepted.

**Why `sum()` across a descriptor's own columns, not `max()`**: an earlier version used
`max()`, which credits a wide descriptor (e.g. `PEOE_VSA`, 14 columns) only for its single
best-performing column, while a scalar descriptor's entire gain sits undivided in its one
column — `max()` structurally underrates a wide block whose real signal is spread across
several correlated columns. This was empirically confirmed to be exactly what was
happening: with `max()`, the N≤5 tail of this trace eliminated `PEOE_VSA`/`SlogP_VSA` —
the two strongest descriptors in the whole candidate pool by standalone R² (§8b) — before
much weaker scalars, producing a spurious floor at N=2 (R²=0.066, barely above noise).
Switching to `sum()` (`src/feature_selection/feature_selection.py::run_descriptor_rfe`)
fixes this — see §6/§9 for the corrected trace, and §8b's max-vs-sum comparison table for
the full before/after.

**`MIN_DESCRIPTORS`**: how far down the elimination is allowed to go before stopping.
Set to 1 so the trace runs all the way to a single descriptor, rather than stopping at an
arbitrary floor before the real elbow is visible — an earlier run used `MIN_BLOCKS=10`
picked with no such justification, and a later one stopped at `min_descriptors=3`, cutting
the trace short before the real elbow was visible.

**`CV_FOLDS`**: 5, not 3. 3-fold gave visibly noisy, non-monotonic step-to-step swings in
the earlier run (values zigzagging up and down as descriptors were removed, inconsistent
with a real information-loss trend) — 5-fold trades some speed for a less noisy trace,
which matters here since the whole point of this section is reading the trace by eye.


In [ ]:
rfe_descriptor_map = {name: descriptor_map[name] for name in kept_after_vif}

MIN_DESCRIPTORS = 1  # low on purpose -- see markdown above; we want to see the real elbow, not stop before it
CV_FOLDS = 5  # up from 3 -- see markdown above; 3-fold was too noisy to read the trace by eye

rfe_trace = run_descriptor_rfe(
    X_by_endpoint,
    y_by_endpoint_arr,
    rfe_descriptor_map,
    min_descriptors=MIN_DESCRIPTORS,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
)
joblib.dump(rfe_trace, DATA_PROC / 'section_fs_rfe_trace.pkl')
print(f'RFE trace: {len(rfe_trace)} steps, from {rfe_trace[0]["n_descriptors"]} down to {rfe_trace[-1]["n_descriptors"]} descriptors')

## 6 — Inspect the trace, pick a cutoff

Plot mean CV score (across the 4 endpoints) against the number of surviving descriptors.
Pick the smallest descriptor count before the score visibly drops off.


In [ ]:
# Combine the earlier stage audits (48 post-constant-filter, 27 post-correlation, 16
# post-VIF -- the last of which is also rfe_trace's own first row) with the RFE trace
# (16 -> 1), so the whole pipeline's R^2-vs-descriptors story is one continuous view
# rather than only the RFE sub-range.
combined = [
    {'n_descriptors': len(kept_map), 'cv_score_mean': baseline_r2['mean'], 'stage': 'post constant-filter'},
    {'n_descriptors': len(kept_after_corr), 'cv_score_mean': corr_stage_r2['mean'], 'stage': 'post correlation-prune (CCA)'},
] + [{'n_descriptors': row['n_descriptors'], 'cv_score_mean': row['cv_score_mean'], 'stage': 'RFE'} for row in rfe_trace]

for row in combined:
    print(f"{row['n_descriptors']:3d} descriptors  mean_r2={row['cv_score_mean']:.4f}  ({row['stage']})")

n_descriptors_seq = [row['n_descriptors'] for row in combined]
score_seq = [row['cv_score_mean'] for row in combined]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_descriptors_seq, score_seq, marker='o')
ax.axvline(len(kept_after_corr), color='tab:orange', linestyle=':', linewidth=1.5,
           label=f'canonical-correlation prune (CCA): 48 -> {len(kept_after_corr)}')
ax.axvline(len(kept_after_vif), color='grey', linestyle='--', linewidth=1,
           label=f'VIF prune: {len(kept_after_corr)} -> {len(kept_after_vif)} (RFE starts here)')
ax.axvline(2, color='tab:green', linestyle=':', linewidth=1.5,
           label='RFE elbow N=2 (sum-aggregated gain)')
ax.set_xlabel('Number of descriptors remaining')
ax.set_ylabel('Mean CV R² across endpoints')
ax.set_title('R² vs. descriptors remaining — full pipeline (constant-filter -> CCA correlation-prune -> VIF -> RFE)')
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'section_fs_rfe_trace.png', dpi=150)
plt.show()


In [ ]:
# Chosen after inspecting the trace above: with sum()-aggregated gain (src/feature_selection
# fix -- see §5), R^2 declines gently from 16 down to 2 descriptors (every step within the
# ~0.004-0.011 CV-fold noise floor established in src/feature_selection/CLAUDE.md), then
# hits the one real elbow at 2->1 (-0.081, dropping SlogP_VSA -- the single biggest drop
# anywhere in the trace). 2 descriptors is the smallest set before that cliff.
# This is a change from an earlier run of this notebook, which used max()-aggregated gain
# and picked SELECTED_N_DESCRIPTORS=5 -- that run's tail (N<=5) was later shown (§8b) to be
# unreliable: max() structurally underrated PEOE_VSA/SlogP_VSA (wide descriptors whose gain
# is spread across several columns) relative to scalars, causing RFE to eliminate the two
# strongest descriptors in the whole pool before much weaker ones. With sum(), RFE's own
# trace now reaches directly the same {PEOE_VSA, SlogP_VSA} conclusion that the standalone-
# ranking workaround in §8b previously had to recover separately -- see §9 for the full history.
SELECTED_N_DESCRIPTORS = 2

chosen_row = next(row for row in rfe_trace if row['n_descriptors'] == SELECTED_N_DESCRIPTORS)
selected_descriptors = chosen_row['descriptors']
selected_features = sorted(f for name in selected_descriptors for f in descriptor_map[name])

print(f'Selected {len(selected_descriptors)} descriptors -> {len(selected_features)} of 316 rdmoldes features:')
print(f'  {selected_descriptors}')

print(f'\nFor reference -- the {len(kept_after_corr)} descriptors after correlation-prune (before VIF/RFE):')
print(f'  {kept_after_corr}')

print(f'\nCV R^2 by stage:')
print(f'  {len(kept_map):3d} descriptors, no pruning yet:                    {baseline_r2["mean"]:.4f}')
print(f'  {len(kept_after_corr):3d} descriptors, after canonical-correlation prune (CCA): {corr_stage_r2["mean"]:.4f}')
print(f'  {len(kept_after_vif):3d} descriptors, after VIF prune:                          {vif_stage_r2["mean"]:.4f}')
print(f'  {SELECTED_N_DESCRIPTORS:3d} descriptors, after LightGBM RFE (final cutoff):    {chosen_row["cv_score_mean"]:.4f}')

output = {
    'selected_descriptors': selected_descriptors,
    'selected_features': selected_features,
    'cv_score_mean': chosen_row['cv_score_mean'],
    'cv_score_by_endpoint': chosen_row['cv_score_by_endpoint'],
    'baseline_cv_score_mean': baseline_r2['mean'],
    'post_cca_correlation_prune_cv_score_mean': corr_stage_r2['mean'],
    'post_vif_prune_cv_score_mean': vif_stage_r2['mean'],
    'dropped_constant': dropped_constant,
    'dropped_correlation': corr_dropped,
    'dropped_vif': vif_trace,
}
with open(DATA_PROC / 'selected_descriptors.json', 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nSaved -> {DATA_PROC / "selected_descriptors.json"}')


## 7 — Diagnostic: does real 3D geometry matter? (ADR-011 follow-up)

**ADR-011** (`DECISIONS.md`) found that 11 of `rdmoldes()`'s "3D shape" descriptors are
computed on the flat 2D depiction coordinates in the source SDFs (`Is3D()=False`, all
`z=0`), not real conformers — inherited from the paper's own sdf data. Its original
"limited impact" claim was wrong *at the time this section was first written*:
`CalcEccentricity` and `CalcPMI3` were 2 of the 5 descriptors in that run's final selection.

**Update, after the §5/§8 fixes**: §5's RFE originally eliminated descriptors on
LightGBM's *default* `feature_importances_` (`split`-count, not `gain`), and later (still
within §5) was found to be aggregating a descriptor's own column gains with `max()` rather
than `sum()` — both fixed. Neither `CalcEccentricity` nor `CalcPMI3` survives to the
current final selection anymore (§9): the flat-2D-geometry descriptors were already
eliminated by the VIF-prune stage's candidate pool boundary well before RFE's own tail-end
picks which of `PEOE_VSA`/`SlogP_VSA`/etc. survive. The real-vs-flat splice comparisons
below are kept exactly as designed — they're still a valid, independent check on the wider
48/50-descriptor candidate pool — but the "final selection" comparison specifically is now
a no-op (none of the 11 affected descriptors are in that set anymore, so splicing real
values into it changes nothing, confirmed below at delta=0.0000). Left in place rather than
removed, since it's still the correct thing to check given whatever the current selection
happens to be.

This section is the cheap diagnostic ADR-011 calls for: embed **real** 3D conformers for
the same 3509 unique molecules already used above, recompute all 11 affected descriptors,
splice them into the existing feature matrices in place of the flat values, and re-run
`evaluate_descriptor_set` — no `01.5` retraining, no re-running the pruning pipeline
itself. Two clean, controlled comparisons (same descriptor set, only the values change):
the final selection, and the 48-descriptor post-constant-filter baseline. If
CV R² doesn't move, that's a clean negative result; if it does, a full propagation into
`01.5` (ADR-011 Step 2) is warranted.

**Plus one further check, at the baseline only**: `CalcPBF`/`CalcSpherocityIndex` are
constant (0.0) on the flat data *specifically and only* because of the flat-geometry inheritied from the public data released
— a zero-variance feature provably cannot help a tree-based model, so "48 flat" and "50
flat" are identical by construction. That means "48 real vs. 50 real" (adding these 2
back in, with real values) isn't confounded by anything else changing — it isolates
whether real geometry unlocks value flat geometry made structurally invisible. The same
check isn't run at the final-selection level: the other 9 affected descriptors
aren't in that selection because correlation-prune/VIF/RFE (§3–§5) dropped them for
reasons unrelated to being constant, so there's no equivalent guarantee and adding them
back would confound "real vs. flat" with "more descriptors, tested for the first time."

**Conformer choice**: several of the affected descriptors (`CalcAsphericity`,
`CalcEccentricity`, `CalcPMI1-3`, `CalcRadiusOfGyration`, `CalcInertialShapeFactor`) can
vary across a flexible molecule's accessible conformers — a single arbitrarily-seeded
embedding isn't a representative "real 3D geometry" for such a molecule, only one
snapshot of it. So this section embeds an **ensemble of `N_CONFORMERS` conformers per
molecule** (ETKDGv3, fixed base seed for reproducibility) and computes descriptors on the
**lowest-MMFF-energy** conformer of each — the standard cheap proxy for "the geometry a
molecule is actually most likely to be found in," rather than one arbitrary embedding.

**What this section does *not* test**: whether the full pipeline (§1–§6), re-run from
scratch on real-conformer values for all 11 descriptors, would have made different
correlation-prune/VIF/RFE decisions and ended up with a different selection entirely.
That's a real possibility in principle, but checking it means re-running the whole
pipeline from §1 on real-conformer data, not just this splice-and-compare. Judged
unlikely to change the conclusion given how small every real-conformer delta measured
below already is, but left untested: this notebook selects on the paper's own (flat)
data, and a full real-conformer re-run isn't in scope here.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

# All 11 individual flat-geometry descriptors named in ADR-011. CalcPBF/CalcSpherocityIndex
# are exactly 0.0 on the flat data (Section 1) and get constant-filtered out of every
# selection stage below -- they're computed here anyway so Section 7b can test what adding
# them back in, with real conformer values, would have done to R^2.
AFFECTED_3D_DESCRIPTORS = [
    'CalcPBF', 'CalcSpherocityIndex', 'CalcNPR1', 'CalcNPR2',
    'CalcPMI1', 'CalcPMI2', 'CalcPMI3',
    'CalcAsphericity', 'CalcEccentricity',
    'CalcRadiusOfGyration', 'CalcInertialShapeFactor',
]

N_CONFORMERS = 5  # ensemble size per molecule -- see markdown above for why a single embedding isn't enough
MMFF_MAX_ITERS = 500


def _compute_3d_descriptors(mol, conf_id):
    return {
        'CalcPBF': rdMolDescriptors.CalcPBF(mol, confId=conf_id),
        'CalcSpherocityIndex': rdMolDescriptors.CalcSpherocityIndex(mol, confId=conf_id),
        'CalcNPR1': rdMolDescriptors.CalcNPR1(mol, confId=conf_id),
        'CalcNPR2': rdMolDescriptors.CalcNPR2(mol, confId=conf_id),
        'CalcPMI1': rdMolDescriptors.CalcPMI1(mol, confId=conf_id),
        'CalcPMI2': rdMolDescriptors.CalcPMI2(mol, confId=conf_id),
        'CalcPMI3': rdMolDescriptors.CalcPMI3(mol, confId=conf_id),
        'CalcAsphericity': rdMolDescriptors.CalcAsphericity(mol, confId=conf_id),
        'CalcEccentricity': rdMolDescriptors.CalcEccentricity(mol, confId=conf_id),
        'CalcRadiusOfGyration': rdMolDescriptors.CalcRadiusOfGyration(mol, confId=conf_id),
        'CalcInertialShapeFactor': rdMolDescriptors.CalcInertialShapeFactor(mol, confId=conf_id),
    }


def _embed_lowest_energy_conformer(smiles, seed, num_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS, max_retries=5):
    """Embed an ensemble of conformers (ETKDGv3) and return (mol, lowest-MMFF-energy confId).

    A single arbitrarily-seeded embedding is only one snapshot of a flexible molecule's
    accessible geometries -- num_confs embeddings, MMFF-optimized, keeping the lowest-energy
    one, is the standard cheap proxy for "the geometry the molecule is actually most likely
    to be found in" rather than an arbitrary one. Falls back across a few base seeds if
    embedding produces zero conformers; returns (None, None) if it never succeeds.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.useRandomCoords = True
    cids = []
    for attempt in range(max_retries):
        params.randomSeed = seed + attempt
        cids = list(AllChem.EmbedMultipleConfs(mol_h, numConfs=num_confs, params=params))
        if cids:
            break
    if not cids:
        return None, None
    try:
        energies = AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=max_iters)
        converged = [(cid, e) for cid, (not_converged, e) in zip(cids, energies) if not_converged == 0]
        candidates = converged if converged else list(zip(cids, [e for _, e in energies]))
        best_cid = min(candidates, key=lambda t: t[1])[0]
    except Exception:
        best_cid = cids[0]  # optimization failed outright -- fall back to the first embedded conformer
    return mol_h, best_cid


# unique_smiles ordered by gid, matching X_all's row order from section 1
unique_smiles = [None] * len(smiles_to_gid)
for smi, gid in smiles_to_gid.items():
    unique_smiles[gid] = smi

new_desc_by_smiles = {}
embedding_failures = []
for smi in unique_smiles:
    mol_h, best_cid = _embed_lowest_energy_conformer(smi, seed=RANDOM_STATE)
    if mol_h is None:
        embedding_failures.append(smi)
        continue
    new_desc_by_smiles[smi] = _compute_3d_descriptors(mol_h, best_cid)

print(f'Embedded {N_CONFORMERS}-conformer ensembles for {len(new_desc_by_smiles)}/{len(unique_smiles)} unique molecules, '
      f'descriptors computed on each one\'s lowest-MMFF-energy conformer '
      f'({len(embedding_failures)} embedding failures -- their original flat-derived values are kept as fallback, not dropped)')

In [ ]:
# Splice the recomputed values into each endpoint's own feature matrix (X_by_endpoint --
# real per-endpoint rows, not the deduplicated X_all), in place of the flat-derived values,
# for all 11 AFFECTED_3D_DESCRIPTORS -- including CalcPBF/CalcSpherocityIndex, which are
# constant (0.0) on the flat data and never part of kept_map, but are still spliced here so
# X_by_endpoint_3d can answer "would adding them back in, with real values, have helped"
# further down (Section 7), not just "does swapping already-kept descriptors' values matter".
# Molecules with a failed embedding keep their original flat value untouched.
descriptors_to_splice = AFFECTED_3D_DESCRIPTORS
print(f'Splicing {len(descriptors_to_splice)} descriptors: {descriptors_to_splice}')

X_by_endpoint_3d = {ep: X.copy() for ep, X in X_by_endpoint.items()}
for ep in ENDPOINTS_MODEL:
    for i, smi in enumerate(feat[ep]['smiles']):
        if smi not in new_desc_by_smiles:
            continue  # embedding failed for this molecule -- flat value stays
        new_vals = new_desc_by_smiles[smi]
        for name in descriptors_to_splice:
            col = descriptor_map[name][0]  # each is a 1-feature scalar descriptor
            X_by_endpoint_3d[ep][i, col] = new_vals[name]


In [ ]:
# Re-run evaluate_descriptor_set with real-conformer values spliced in, at two clean
# flat-vs-real comparisons (same descriptor set, only the values change): the final
# selection, and the 48-descriptor post-constant-filter baseline.
#
# Plus a third, at N=50: CalcPBF/CalcSpherocityIndex are constant (0.0) on flat data
# specifically and only because of the flat-geometry bug -- a zero-variance feature
# provably cannot help a tree-based model, so "48 flat" and "50 flat" should be identical
# by construction. Verified below rather than just asserted (delta = 0.0 exactly). That
# means "50 flat vs. 50 real" is a clean, unconfounded comparison -- same descriptor set,
# only the values change, same as the other two. (A similar "+9 missing descriptors" check
# at the 5-descriptor selection level was considered and dropped: those 9 were pruned by
# correlation/VIF/RFE for reasons unrelated to being constant, so there's no equivalent
# "flat and real are identical" guarantee there, and the comparison would be confounded.)
dropped_3d_descriptors = [d for d in AFFECTED_3D_DESCRIPTORS if d not in kept_map]

selection_5desc_r2_3d = evaluate_descriptor_set(X_by_endpoint_3d, y_by_endpoint_arr, descriptor_map, selected_descriptors, random_state=RANDOM_STATE)
baseline_r2_3d = evaluate_descriptor_set(X_by_endpoint_3d, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()), random_state=RANDOM_STATE)
baseline_50desc_flat_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()) + dropped_3d_descriptors, random_state=RANDOM_STATE)
baseline_50desc_r2_3d = evaluate_descriptor_set(X_by_endpoint_3d, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()) + dropped_3d_descriptors, random_state=RANDOM_STATE)

# Per-endpoint R^2 only (mean already reported separately in every print/dict below) --
# named once here instead of re-writing the same {k: v for k, v ... if k != "mean"}
# comprehension inline at each of the six call sites further down.
def per_endpoint(r2):
    return {k: v for k, v in r2.items() if k != 'mean'}

selection_5desc_by_ep = per_endpoint(selection_5desc_r2_3d)
baseline_r2_3d_by_ep = per_endpoint(baseline_r2_3d)
baseline_r2_by_ep = per_endpoint(baseline_r2)
baseline_50desc_r2_3d_by_ep = per_endpoint(baseline_50desc_r2_3d)

print(f'Final {SELECTED_N_DESCRIPTORS}-descriptor selection {selected_descriptors}:')
print(f'  flat (original):  mean R^2 = {chosen_row["cv_score_mean"]:.4f}  {chosen_row["cv_score_by_endpoint"]}')
print(f'  real conformers:  mean R^2 = {selection_5desc_r2_3d["mean"]:.4f}  {selection_5desc_by_ep}')
print(f'  delta: {selection_5desc_r2_3d["mean"] - chosen_row["cv_score_mean"]:+.4f}')

print(f'\n{len(kept_map)}-descriptor post-constant-filter baseline:')
print(f'  flat (original):  mean R^2 = {baseline_r2["mean"]:.4f}  {baseline_r2_by_ep}')
print(f'  real conformers:  mean R^2 = {baseline_r2_3d["mean"]:.4f}  {baseline_r2_3d_by_ep}')
print(f'  delta: {baseline_r2_3d["mean"] - baseline_r2["mean"]:+.4f}')

print(f'\n{len(kept_map) + len(dropped_3d_descriptors)}-descriptor baseline (48 + CalcPBF/CalcSpherocityIndex):')
print(f'  flat:             mean R^2 = {baseline_50desc_flat_r2["mean"]:.4f}  (vs. 48-flat: {baseline_50desc_flat_r2["mean"] - baseline_r2["mean"]:+.4f}, expected ~0.0000)')
print(f'  real conformers:  mean R^2 = {baseline_50desc_r2_3d["mean"]:.4f}  {baseline_50desc_r2_3d_by_ep}')
print(f'  delta: {baseline_50desc_r2_3d["mean"] - baseline_50desc_flat_r2["mean"]:+.4f}')

diagnostic_output = {
    'n_unique_molecules': len(unique_smiles),
    'n_embedding_failures': len(embedding_failures),
    'descriptors_spliced': descriptors_to_splice,
    'selection_5desc': {
        'descriptors': selected_descriptors,
        'flat_cv_score_mean': chosen_row['cv_score_mean'],
        'flat_cv_score_by_endpoint': chosen_row['cv_score_by_endpoint'],
        'real_conformer_cv_score_mean': selection_5desc_r2_3d['mean'],
        'real_conformer_cv_score_by_endpoint': selection_5desc_by_ep,
        'delta': selection_5desc_r2_3d['mean'] - chosen_row['cv_score_mean'],
    },
    'baseline_48desc': {
        'descriptors': list(kept_map.keys()),
        'flat_cv_score_mean': baseline_r2['mean'],
        'flat_cv_score_by_endpoint': baseline_r2_by_ep,
        'real_conformer_cv_score_mean': baseline_r2_3d['mean'],
        'real_conformer_cv_score_by_endpoint': baseline_r2_3d_by_ep,
        'delta': baseline_r2_3d['mean'] - baseline_r2['mean'],
    },
    'baseline_50desc': {
        'descriptors': list(kept_map.keys()) + dropped_3d_descriptors,
        'flat_cv_score_mean': baseline_50desc_flat_r2['mean'],
        'flat_cv_score_by_endpoint': per_endpoint(baseline_50desc_flat_r2),
        'real_conformer_cv_score_mean': baseline_50desc_r2_3d['mean'],
        'real_conformer_cv_score_by_endpoint': baseline_50desc_r2_3d_by_ep,
        'delta': baseline_50desc_r2_3d['mean'] - baseline_50desc_flat_r2['mean'],
    },
}
with open(DATA_PROC / 'adr011_3d_diagnostic.json', 'w') as f:
    json.dump(diagnostic_output, f, indent=2)
print(f'\nSaved -> {DATA_PROC / "adr011_3d_diagnostic.json"}')


In [ ]:
# Report-ready table: R^2 at each stage. Per-endpoint columns included -- descriptors
# are a property of molecular structure, not of a specific endpoint, so a stage that
# looks cheap on the mean but costly for one endpoint is worth catching here rather than
# only after downstream per-endpoint models are trained (this is exactly how the MDR1
# real-conformer regression shows up).
#
# Two separate delta columns, since "cost of a pruning technique" and "cost of flat vs.
# real conformer values" are different questions and a single column can't cleanly answer
# both: delta_from_prev_technique is populated for the constant-filter/CCA/VIF/RFE stages
# and the hand-picked N=3 reference row; delta_real_vs_flat is populated only for the
# three real-conformer rows, each compared against its own same-descriptor-set flat
# counterpart. Each row only ever has one of the two populated -- the other is blank.
manual_3desc = ['CalcTPSA', 'PEOE_VSA', 'SlogP_VSA']
manual_3desc_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, manual_3desc, random_state=RANDOM_STATE)

n_50desc = len(kept_map) + len(dropped_3d_descriptors)

# Each stage is a dict rather than a positional tuple so the fields below are self-labelled
# at the point of use -- no separate legend comment needed to decode a row.
stage_defs = [
    {'stage': f'-- {n_50desc}-descriptor baseline, real conformers spliced in', 'n_descriptors': n_50desc,
     'scores': baseline_50desc_r2_3d, 'technique_prev': None, 'real_vs_flat_prev': baseline_50desc_flat_r2['mean']},
    {'stage': f'-- {n_50desc}-descriptor baseline, flat (verifies "flat {len(kept_map)} == flat {n_50desc}")', 'n_descriptors': n_50desc,
     'scores': baseline_50desc_flat_r2, 'technique_prev': baseline_r2['mean'], 'real_vs_flat_prev': None},
    {'stage': 'Post constant-filter (start)', 'n_descriptors': len(kept_map),
     'scores': baseline_r2, 'technique_prev': None, 'real_vs_flat_prev': None},
    {'stage': f'-- {len(kept_map)}-descriptor baseline, real conformers spliced in', 'n_descriptors': len(kept_map),
     'scores': baseline_r2_3d, 'technique_prev': None, 'real_vs_flat_prev': baseline_r2['mean']},
    {'stage': 'Post correlation-prune (CCA)', 'n_descriptors': len(kept_after_corr),
     'scores': corr_stage_r2, 'technique_prev': baseline_r2['mean'], 'real_vs_flat_prev': None},
    {'stage': 'Post VIF-prune', 'n_descriptors': len(kept_after_vif),
     'scores': vif_stage_r2, 'technique_prev': corr_stage_r2['mean'], 'real_vs_flat_prev': None},
    {'stage': f'-- hand-picked N=3 ({", ".join(manual_3desc)})', 'n_descriptors': 3,
     'scores': manual_3desc_r2, 'technique_prev': vif_stage_r2['mean'], 'real_vs_flat_prev': None},
    {'stage': f'RFE elbow, N={SELECTED_N_DESCRIPTORS} ({", ".join(selected_descriptors)})', 'n_descriptors': SELECTED_N_DESCRIPTORS,
     'scores': dict(chosen_row['cv_score_by_endpoint'], mean=chosen_row['cv_score_mean']),
     'technique_prev': vif_stage_r2['mean'], 'real_vs_flat_prev': None},
    {'stage': f'-- same {SELECTED_N_DESCRIPTORS} descriptors, real conformers spliced in', 'n_descriptors': SELECTED_N_DESCRIPTORS,
     'scores': selection_5desc_r2_3d, 'technique_prev': None, 'real_vs_flat_prev': chosen_row['cv_score_mean']},
]

stage_cost_rows = []
for stage_def in stage_defs:
    scores = stage_def['scores']
    row = {
        'stage': stage_def['stage'],
        'n_descriptors': stage_def['n_descriptors'],
        'mean_cv_r2': scores['mean'],
        'delta_from_prev_technique': None if stage_def['technique_prev'] is None else scores['mean'] - stage_def['technique_prev'],
        'delta_real_vs_flat': None if stage_def['real_vs_flat_prev'] is None else scores['mean'] - stage_def['real_vs_flat_prev'],
    }
    for ep in ENDPOINTS_MODEL:
        row[f'{ep}_cv_r2'] = scores[ep]
    stage_cost_rows.append(row)

stage_cost_df = pd.DataFrame(stage_cost_rows)
stage_cost_df.to_csv(DATA_PROC / 'section_fs_pruning_stage_costs.csv', index=False)
pd.set_option('display.max_colwidth', None)  # stage labels run past pandas' default 50-char truncation
stage_cost_df


## 8 — Diagnostic: gain-importance shadowing/masking

**Purpose**: RFE (§5) trusts LightGBM gain-importance to decide what's safe to drop. If
two descriptors are redundant in a way trees can't see, RFE can keep the wrong one purely
by chance of elimination order — undermining the final selection without showing up as an
R² drop, since the model still has an informationally-equivalent descriptor available. This
section checks whether that actually happened for the current selection, and if so, whether
it changes anything (it doesn't move R² either way, but it does mean "this descriptor is
irreplaceable" isn't a safe reading of the final set — see §9).

`run_descriptor_rfe` eliminates using LightGBM gain-importance — how much a descriptor's
splits reduced loss, summed over the model. **This section is how that "gain" claim got
audited in the first place, and it caught a real bug**: `run_descriptor_rfe` was actually
eliminating on scikit-learn's *default* `feature_importances_`, which for `LGBMRegressor`
is `split`-count, not `gain` — a mismatch between what every comment/docstring here
described and what the code did. The smoking gun was exactly the shadowing case this
section is designed to catch: `CalcEccentricity` (in that run's final selection) had
**zero** importance in the full 316-feature model across all 4 endpoints, while its
near-perfect rank-twin `CalcNPR2` (Spearman ρ=1.0, dropped at correlation-prune, §3) had
real, nonzero importance — `CalcEccentricity` survived RFE to that run's cutoff purely
because split-count importance happened to route all the credit to `CalcNPR2` instead,
before `CalcNPR2` itself got dropped for an unrelated (linear-correlation) reason.

**Fixed** (`importance_type='gain'` set explicitly in `run_descriptor_rfe` and the RF/LightGBM
comparison below) and the whole pipeline re-run. The severe case is gone — `CalcEccentricity`
and `CalcNPR2` are no longer in play at all, since neither made it into the corrected
selection (§9). What's left to check, on genuine gain-importance this time: is there still
*any* near-rank-identical pair split across kept/dropped, and does swapping it change anything.
There is one, milder case — see the swap test below — which is the expected, not alarming,
outcome: CCA/VIF are still linear checks, so some tree-invisible-but-real redundancy getting
past them isn't a bug, it's exactly the residual risk this section exists to catch on every run.

**A second, distinct gain-importance bug was found later** (still within §5, after this
section's original run): `run_descriptor_rfe` was aggregating a descriptor's own column
gains with `max()` — crediting a wide descriptor (like `PEOE_VSA`, 14 columns) only for its
single best-performing column, systematically underrating it relative to a scalar whose
entire gain sits undivided in one column. That's a different failure mode from the
shadowing this section targets (this section is about *redundancy between* descriptors;
that one was about *unfair aggregation within* a descriptor), fixed separately by switching
to `sum()` — see §5 and §8b for the full before/after.

Three checks, generalised across the whole §4 candidate pool (not hardcoded to this one
pair) so this is a reusable diagnostic, not a one-off:

**Current outcome**: both descriptors in the final selection (`PEOE_VSA`, `SlogP_VSA`)
are multi-feature vector descriptors, and this section's rank-correlation/swap-test checks
(1 and 3 below) only ever apply to *scalar* (1-feature) descriptors — a scalar-vs-scalar
Spearman rho is a single well-defined number the way a scalar-vs-vector or vector-vs-vector
comparison isn't. So on the current selection, checks 1 and 3 are a structural no-op (no
scalar descriptors to check), not a null result — this is expected given the current
selection's composition, not evidence the shadowing risk has gone away for some future
selection that does include scalar descriptors. Check 2 (RF vs. LightGBM importance
divergence) is also scalar-only as implemented, so it's a no-op here too — see the code
cell below.

1. **Rank-correlation redundancy CCA/VIF may have missed** — CCA (§3) and VIF (§4) are
   both fundamentally *linear* checks. Two descriptors can be only moderately correlated
   linearly (canonical r < 0.9, passing both checks) while being **exactly rank-identical**
   (Spearman ρ=1.0) — informationally indistinguishable to any *tree* model (which only
   ever asks "which side of a threshold"), even though a smooth model like FCNN could
   still tell them apart.
2. **RF vs. LightGBM importance divergence** — do the two tree ensembles even agree on
   which descriptors matter, in the full 316-feature set? If not, LightGBM-only RFE is a
   biased view of "what these models find important" (FCNN excluded from this specific
   check — no native importance metric, would need permutation importance and a trained
   model.
3. **Direct swap test** — for any §6-selected descriptor with a near-rank-identical
   relative among the descriptors §3/§4 dropped, does swapping one for the other change CV
   R² at all? A near-zero delta confirms the descriptor's presence in the final set is
   substitutable, not uniquely load-bearing.


In [ ]:
from scipy.stats import spearmanr

# For every descriptor still in the final selection, find its most rank-correlated
# relative among ALL earlier-dropped scalar descriptors (constant/correlation/VIF/RFE),
# using the deduplicated pool X_all. Only scalar (1-feature) descriptors are checked here
# -- rank correlation between a scalar and a multi-feature vector descriptor isn't a
# single well-defined number the way it is between two scalars.
all_dropped_names = dropped_constant + [r['dropped'] for r in corr_dropped] + [r['dropped'] for r in vif_trace] + \
    [name for name in kept_after_vif if name not in selected_descriptors]
scalar_dropped = [name for name in all_dropped_names if len(descriptor_map[name]) == 1]

shadow_candidates = {}
for name in selected_descriptors:
    if len(descriptor_map[name]) != 1:
        continue  # skip multi-feature descriptors (PEOE_VSA, SlogP_VSA) -- same reasoning as above
    target = X_all[:, descriptor_map[name][0]]
    best_rho, best_partner = 0.0, None
    for other in scalar_dropped:
        rho, _ = spearmanr(target, X_all[:, descriptor_map[other][0]])
        if abs(rho) > abs(best_rho):
            best_rho, best_partner = rho, other
    shadow_candidates[name] = (best_partner, best_rho)
    flag = ' <-- NEAR RANK-IDENTICAL (tree-invisible redundancy)' if abs(best_rho) > 0.95 else ''
    print(f'{name:20s} most rank-correlated with {best_partner or "(none found)":26s}  Spearman rho={best_rho:+.4f}{flag}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Fit LightGBM and RF on the FULL 316-feature set (not the pruned selection) for each
# endpoint, then compare importance for every final-selection descriptor plus its shadow
# candidate identified above -- shows whether the two model types even agree on which
# descriptors matter, before any pruning has happened.
# Scalar-only, same reasoning as the shadow-candidate search above (§8 markdown) -- a
# multi-feature descriptor's importance would need summing across its own features to
# compare fairly, skip for now. On the current selection both final descriptors are
# multi-feature (PEOE_VSA, SlogP_VSA), so this comparison has nothing scalar to plot.
compare_names = sorted(set(selected_descriptors) | {p for p, _ in shadow_candidates.values() if p is not None})

lgbm_imp_by_ep, rf_imp_by_ep = {}, {}
for ep in ENDPOINTS_MODEL:
    X = feat[ep]['rdkit']
    y = feat[ep]['y']
    lgbm = LGBMRegressor(n_estimators=300, random_state=RANDOM_STATE, verbose=-1, importance_type='gain').fit(X, y)
    rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(X, y)
    lgbm_imp_by_ep[ep] = lgbm.feature_importances_ / lgbm.feature_importances_.sum()
    rf_imp_by_ep[ep] = rf.feature_importances_ / rf.feature_importances_.sum()

# Mean normalised importance across the 4 endpoints, single-feature descriptors only.
lgbm_mean = {name: np.mean([lgbm_imp_by_ep[ep][descriptor_map[name][0]] for ep in ENDPOINTS_MODEL])
             for name in compare_names if len(descriptor_map[name]) == 1}
rf_mean = {name: np.mean([rf_imp_by_ep[ep][descriptor_map[name][0]] for ep in ENDPOINTS_MODEL])
           for name in compare_names if len(descriptor_map[name]) == 1}

plot_names = list(lgbm_mean.keys())

if not plot_names:
    print(f'No scalar descriptors among {compare_names} (final selection + shadow candidates) -- '
          'nothing to plot. This is expected when the final selection is entirely multi-feature '
          'descriptors (see §8 markdown); the chart is skipped rather than saved blank.')
else:
    x = np.arange(len(plot_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    bars_lgbm = ax.bar(x - width/2, [lgbm_mean[n] for n in plot_names], width, label='LightGBM', color='tab:blue')
    bars_rf = ax.bar(x + width/2, [rf_mean[n] for n in plot_names], width, label='RandomForest', color='tab:orange')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_names, rotation=30, ha='right')
    ax.set_ylabel('Mean normalised feature importance (share of total, 4-endpoint average)')
    ax.set_title('LightGBM vs. RF importance, full 316-feature model — final-selection descriptors + shadow candidates')
    for name in plot_names:
        if name in selected_descriptors:
            ax.get_xticklabels()[plot_names.index(name)].set_fontweight('bold')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / 'section_fs_importance_shadowing.png', dpi=150)
    plt.show()

    print(f'(bold x-labels = in the final {len(selected_descriptors)}-descriptor selection; others are their shadow candidates)')

pd.DataFrame({'lightgbm_importance': lgbm_mean, 'rf_importance': rf_mean})


In [ ]:
# For every final-selection descriptor flagged as near-rank-identical (|rho|>0.95) to a
# dropped relative, swap it out for that relative and re-measure CV R^2 on the final
# selection -- a near-zero delta confirms the descriptor is substitutable, not uniquely
# load-bearing (as opposed to e.g. SlogP_VSA, whose removal cost -0.076 in the RFE trace, §6).
final_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, selected_descriptors, random_state=RANDOM_STATE)
print(f'Baseline -- final {len(selected_descriptors)}-descriptor selection: mean R^2 = {final_r2["mean"]:.4f}\n')

swap_results = []
for name, (partner, rho) in shadow_candidates.items():
    if partner is None or abs(rho) <= 0.95:
        continue
    swapped_set = [partner if d == name else d for d in selected_descriptors]
    swapped_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, swapped_set, random_state=RANDOM_STATE)
    delta = swapped_r2['mean'] - final_r2['mean']
    swap_results.append({'original': name, 'swapped_in': partner, 'spearman_rho': rho,
                          'r2_original': final_r2['mean'], 'r2_swapped': swapped_r2['mean'], 'delta': delta})
    print(f'{name} -> {partner}  (rho={rho:+.4f}):  R^2 {final_r2["mean"]:.4f} -> {swapped_r2["mean"]:.4f}  (delta={delta:+.4f})')

if not swap_results:
    print('No final-selection descriptor has a |rho|>0.95 dropped relative -- nothing to swap-test.')
else:
    swap_df = pd.DataFrame(swap_results)
    swap_df.to_csv(DATA_PROC / 'section_fs_shadowing_swap_test.csv', index=False)
    print(f'\nSaved -> {DATA_PROC / "section_fs_shadowing_swap_test.csv"}')

## 8b — Standalone contribution: confirmatory check on RFE's tail-end order

**Distinct question from §8.** §8 asked whether a *survivor* is redundant with something
already dropped (`CalcEccentricity` vs. `CalcNPR2`) — a question about redundancy among
descriptors still in the set. This section asks a narrower, different question: once RFE
got down to its last few descriptors, did gain-importance keep dropping the *least
valuable* one, or just the one LightGBM happened to lean on least at that step, given
whatever else was already in the model?

**Historical note**: this section was originally written as a *correction* — with the
old `max()`-aggregated gain-importance (§5), RFE's own tail-end trace dropped
`PEOE_VSA`/`SlogP_VSA` (the two strongest descriptors in the pool, standalone) before much
weaker scalars, and this section's standalone-ranking method had to be used to recover the
right answer separately. After switching RFE to `sum()`-aggregated gain (§5), RFE's own
trace now reaches the *same* conclusion on its own (§6/§9) — so this section is no longer
a required correction, just a **confirmatory diagnostic**: a second, independent method
(single-descriptor CV R², no RFE involved at all) that should agree with whatever RFE's
own trace says, as a standing check against this class of proxy-metric distortion
recurring in the future (e.g. if the descriptor pool or featurization changes).

**Method**: evaluate each of the 16 post-VIF candidate-pool descriptors completely on
its own (CV R² of a single-descriptor model) -- the whole pool RFE started from, not just
the descriptors that happened to survive to the final selection, so there are enough
ranked candidates to build a genuine top-N set at every N checked below. Then, for a few
descriptor counts, compare RFE's own trace step
against the set of the same size built by keeping whichever descriptors scored highest
*standalone* — not another RFE run, just a direct sanity check that the descriptors RFE
kept outscore the descriptors RFE dropped, at each size.


In [ ]:
# Evaluated over the whole 16-descriptor RFE candidate pool (kept_after_vif), not just the
# final selection -- needed so the N=5/4/3/2 confirmatory comparison below (§8b's own
# purpose) has enough ranked candidates to build a genuine top-N-by-standalone-R^2 set at
# each N, rather than only ever having the 2 finally-selected descriptors to draw from.
standalone_r2 = {}
for name in kept_after_vif:
    standalone_r2[name] = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, [name], random_state=RANDOM_STATE)

standalone_df = pd.DataFrame({
    name: {'mean_cv_r2': r2['mean'], **{ep: r2[ep] for ep in ENDPOINTS_MODEL}}
    for name, r2 in standalone_r2.items()
}).T.sort_values('mean_cv_r2', ascending=False)
standalone_df.to_csv(DATA_PROC / 'section_fs_standalone_r2.csv')
standalone_df


In [ ]:
# Confirmatory check (see §8b markdown): now that RFE uses sum()-aggregated gain, its own
# trace should already agree closely with the standalone-ranked (top-N by solo R^2) set at
# every N -- unlike the old max()-aggregated trace, which diverged sharply below N=5 (e.g.
# its own N=2 pick, {CalcTPSA, CalcChi3v}, scored R^2=0.066 vs. 0.346 for the standalone-
# ranked pair). This cell re-runs that same comparison on the corrected trace.
ranked_by_standalone = standalone_df.index.tolist()

rfe_vs_topn_solo_rows = []
for n in [5, 4, 3, 2]:
    rfe_step = next(row for row in rfe_trace if row['n_descriptors'] == n)
    topn_solo = ranked_by_standalone[:n]
    topn_solo_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, topn_solo, random_state=RANDOM_STATE)
    rfe_vs_topn_solo_rows.append({
        'n_descriptors': n,
        'rfe_own_set': rfe_step['descriptors'],
        'rfe_own_r2': rfe_step['cv_score_mean'],
        'topn_solo_set (top-N by standalone R^2)': topn_solo,
        'topn_solo_r2': topn_solo_r2['mean'],
        'delta': topn_solo_r2['mean'] - rfe_step['cv_score_mean'],
    })

rfe_vs_topn_solo_df = pd.DataFrame(rfe_vs_topn_solo_rows)
rfe_vs_topn_solo_df.to_csv(DATA_PROC / 'section_fs_rfe_vs_topn_solo.csv', index=False)
rfe_vs_topn_solo_df


## 9 — Takeaway

**Note on this section**: two real bugs in `run_descriptor_rfe` were found and fixed while
building this notebook, both changing the final selection:

1. **Split-count vs. gain-importance.** §5's RFE was originally eliminating descriptors on
   LightGBM's *default* `feature_importances_` (`split`-count), not the `gain`-based
   importance every comment/docstring described. Fixed (`importance_type='gain'`, set
   explicitly). This changed the selection from a 5-descriptor set that included
   `CalcEccentricity`/`CalcPMI3` (ADR-011's flat-2D-geometry descriptors) to one with
   `CalcNumAromaticCarbocycles`/`CalcChi3v` instead.
2. **`max()` vs. `sum()` block-gain aggregation.** Even on genuine gain-importance, RFE's
   tail end (N≤5) was still unreliable: descriptor importance was `max()` over a
   descriptor's own feature columns, which credits a wide descriptor (`PEOE_VSA`, 14
   columns; `SlogP_VSA`, 12 columns) only for its single best-performing column, while a
   scalar descriptor's entire gain sits undivided in one column — `max()` structurally
   underrated the two widest, and (per §8b) individually *strongest*, descriptors in the
   whole pool. §8b's standalone-ranking workaround had to be built specifically to recover
   the right answer despite this. Fixed by aggregating with `sum()` instead — see §5 and
   `src/feature_selection/CLAUDE.md` for the full before/after trace comparison. Every
   number and descriptor name below is from the pipeline re-run after **both** fixes.

**Is there a small, stable descriptor set that gets you most of the full-featureset
performance?** Yes:

| Stage | Descriptors | Features | Mean CV R² | % of 48-descriptor ceiling |
|---|---|---|---|---|
| No pruning | 48 | 314 | 0.4193 | 100% |
| Post correlation-prune (CCA) | 27 | 92 | 0.4057 | 97% |
| Post VIF-prune | 16 | 40 | 0.3824 | 91% |
| RFE trace, N=5 (`CalcTPSA`, `CalcNumAromaticCarbocycles`, `CalcChi3v`, `PEOE_VSA`, `SlogP_VSA`) | 5 | 29 | 0.3606 | 86% |
| RFE trace, N=3 | 3 | 27 | 0.3437 | 82% |
| **RFE trace, §6 elbow (N=2): `PEOE_VSA`, `SlogP_VSA`** | **2** | **26** | **0.3458** | **82%** |
| RFE trace, N=1 (`SlogP_VSA` alone) | 1 | 12 | 0.2651 | 63% |

With `sum()`-aggregated gain, RFE's own trace is now **flat from N=5 down to N=2** — every
step in that range is within the ~0.004–0.011 CV-fold noise floor established in
`src/feature_selection/CLAUDE.md` — with a single real elbow at N=2→1 (−0.081, dropping
`SlogP_VSA`, the largest drop anywhere in the trace). **§8b confirms this directly**: at
every N checked (5/4/3/2), RFE's own greedy trace now lands within noise of the
standalone-ranked (top-N by solo R²) set of the same size — unlike the old `max()`-based
trace, whose own N=2 pick (`CalcTPSA` + `CalcChi3v`) scored R²=0.066, barely above noise,
versus 0.346 for the standalone-ranked pair. §8b is no longer a required correction — it's
a confirmatory diagnostic that agrees with what RFE finds on its own.

**What N makes sense?** **N=2 (`PEOE_VSA`, `SlogP_VSA`) is the selected set** — not a
hand-wavy compromise, but RFE's own elbow: it captures 82% of the full 48-descriptor
ceiling and 96% of what the 5-descriptor set (RFE's N=5 point, 86%) achieves, with 26 of
that set's 29 features. Every descriptor added past it buys single-digit-R²-point returns
at best (N=5 over N=2: +0.015 R² for 3 more descriptors), and the trace shows those
descriptors (`CalcTPSA`, `CalcNumAromaticCarbocycles`, `CalcChi3v`) are individually weak
(solo R²=0.038–0.06, §8b) — present in the wider sets mostly as noise-level filler, not
distinct signal. If the extra ~0.015 R² is worth 3 more descriptors for a given downstream
model, the N=5 point remains a defensible, fully-computed alternative (`rfe_trace`,
`data/processed/section_fs_rfe_trace.pkl`) — it just isn't the recommended default anymore.

**What do these descriptors physically represent?**

- **N=2 (selected): `PEOE_VSA` + `SlogP_VSA`.** Both are "VSA" descriptors — total van der
  Waals surface area, binned into ranges of an atomic property. `PEOE_VSA` bins by
  Gasteiger partial charge (14 bins): how much of the molecule's surface is
  electron-rich vs. electron-poor — its polarity/hydrogen-bonding character.
  `SlogP_VSA` bins by each atom's Crippen logP contribution (12 bins): how much surface
  is lipophilic ("greasy") vs. not, and where. Together they're a compact
  "how much polar surface, how much lipophilic surface" fingerprint — the two most
  fundamental physicochemical axes for how a molecule partitions between aqueous and
  lipid/protein environments. That's directly mechanistic for all 4 endpoints modelled
  here: HLM/RLM metabolic clearance (driven by lipophilicity plus accessible polar
  sites for oxidation), MDR1 efflux recognition (transporter binding driven by
  amphipathic surface patterns), and aqueous solubility (a direct polarity-vs-lipophilicity
  balance) are all textbook lipophilicity/polarity-governed ADME properties — consistent
  with VSA descriptors topping feature-importance lists across the ADME QSAR literature,
  not a dataset-specific fluke.
- **N=5 (RFE's own trace, if more descriptors are wanted): adds `CalcTPSA`,
  `CalcNumAromaticCarbocycles`, `CalcChi3v`.** `CalcTPSA` (topological polar surface area)
  is a single global number summing polar (N/O and attached H) surface from 2D topology —
  a coarser, complementary summary of polarity alongside `PEOE_VSA`'s 14-bin breakdown,
  and the classic descriptor behind Lipinski/Veber-style permeability rules.
  `CalcNumAromaticCarbocycles` counts benzene-type aromatic rings — a structural proxy for
  planarity, lipophilicity, and common sites of oxidative metabolism. `CalcChi3v` is a
  3rd-order valence connectivity index (Kier-Hall), a graph-theoretic branching/size proxy —
  the weakest of the 5 standalone (§8b, R²=0.040).
- **N=16 (post-VIF), by descriptor family**: `PEOE_VSA`/`SlogP_VSA` (surface polarity +
  lipophilicity, as above); polarity/H-bonding counts (`CalcTPSA`, `CalcNumLipinskiHBD`,
  `CalcNumHeteroatoms`, `CalcNumAmideBonds`); ring/scaffold composition
  (`CalcNumAliphaticHeterocycles`, `CalcNumAromaticCarbocycles`, `CalcNumAromaticRings`,
  `CalcNumSaturatedRings`); flexibility/size (`CalcNumRotatableBonds`, `CalcChi3v`); and
  4 flat-2D-geometry descriptors carried over from ADR-011's affected set
  (`CalcEccentricity`, `CalcInertialShapeFactor`, `CalcPMI1`, `CalcPMI3`) — none of which
  make it into the final selection, consistent with §7's finding that real 3D conformers
  don't move R² much for this dataset. Broadly: this 16-descriptor set is a standard ADME
  descriptor palette (polarity, lipophilicity, ring/scaffold makeup, flexibility, shape),
  just redundant enough internally that VIF/RFE compress most of its signal into 2
  descriptors without much loss.


## 9b — Variance-explained-vs-feature-count plot

The table above, as a chart: mean CV R² (variance explained) against feature count, one
bar per **real pipeline stage** (no pruning -> correlation-prune -> VIF-prune -> RFE
elbow). Bars are annotated with descriptor count and feature count so the cost side of
the tradeoff is visible alongside the R² side.

An earlier version of this chart added two extra bars built from §8b's standalone-ranking
workaround ("Top-3 by solo R²", "Top-2 by solo R²"), because the old `max()`-aggregated
RFE trace didn't reach a good N=2/N=3 set on its own. Now that RFE uses `sum()`-aggregated
gain (§5) and its own trace lands on the same {`PEOE_VSA`, `SlogP_VSA`} set directly (§9),
those extra bars are redundant with the "RFE elbow" bar and have been removed — §8b is
kept as a confirmatory diagnostic (its numbers now agree with this chart's last bar) rather
than a second source of candidate sets.


In [ ]:
stage_rows = [
    {'stage': 'No pruning', 'descriptors': list(kept_map.keys()), 'mean_cv_r2': baseline_r2['mean']},
    {'stage': 'Post correlation-\nprune (CCA)', 'descriptors': kept_after_corr, 'mean_cv_r2': corr_stage_r2['mean']},
    {'stage': 'Post VIF-prune', 'descriptors': kept_after_vif, 'mean_cv_r2': vif_stage_r2['mean']},
    {
        'stage': f'RFE elbow\n(N={SELECTED_N_DESCRIPTORS})',
        'descriptors': selected_descriptors,
        'mean_cv_r2': chosen_row['cv_score_mean'],
    },
]

ceiling = baseline_r2['mean']
for row in stage_rows:
    row['n_descriptors'] = len(row['descriptors'])
    row['n_features'] = sum(len(descriptor_map[name]) for name in row['descriptors'])
    row['pct_of_ceiling'] = row['mean_cv_r2'] / ceiling * 100

stage_summary_df = pd.DataFrame(stage_rows)[['stage', 'n_descriptors', 'n_features', 'mean_cv_r2', 'pct_of_ceiling']]
stage_summary_df.to_csv(DATA_PROC / 'section_fs_stage_summary.csv', index=False)
stage_summary_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(stage_summary_df))
bars = ax.bar(x, stage_summary_df['mean_cv_r2'], color='steelblue', edgecolor='white')

for bar, row in zip(bars, stage_summary_df.itertuples()):
    ax.annotate(
        f'{row.n_features} features\n({row.n_descriptors} desc.)',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4), textcoords='offset points', ha='center', fontsize=8,
    )
    ax.annotate(
        f'{row.pct_of_ceiling:.0f}%',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2),
        ha='center', fontsize=8, color='white', fontweight='bold',
    )

ax.axhline(ceiling, color='grey', linestyle='--', linewidth=1, label=f'No-pruning ceiling ({ceiling:.3f})')
ax.set_xticks(x)
ax.set_xticklabels(stage_summary_df['stage'], fontsize=9)
ax.set_ylabel('Mean CV R² across endpoints\n(variance explained)')
ax.set_title('Variance explained vs. feature count, by pruning stage')
ax.set_ylim(0, ceiling * 1.22)
ax.legend(loc='lower left', fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / 'section_fs_variance_explained_by_stage.png', dpi=150)
plt.show()